# Biological BMI Paper — OLS Regressions for Numeric Features

***by Kengo Watanabe***  

In this notebook, baseline BMI or biological BMI is regressed to each baseline numeric feature systematically. Also, each numeric feature is regressed to BMI class in point of misclassification.  

Input:  
* Summary of all biological BMIs: 210105_Biological-BMI-paper_misclassification_biological-BMI-BothSex_summary.csv  
* Cleaned numeric features: 210106_Biological-BMI-paper_data-cleaning_OLS-regression_numDF.tsv  
* Feature label (for presentation): 210727_feature-label-correspondence.csv  

Output:  
* Figure 1c  
* Figure 3c  

Original notebook:  
* 210106_Biological-BMI-paper_OLS-regression_numeric-features.ipynb  
* 220117_Biological-BMI-paper_OLS-regression_numeric-features-ver3.ipynb  

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
#For Arial font
#!conda install -c conda-forge -y mscorefonts
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
from statsmodels.stats import multitest as multi

!conda list

## 1. Preprocessing for OLS regression

### 1-1. Prepare dataset

In [ ]:
#Import BMI and biological BMI
fileDir = './ExportData/'
fileName = '210105_Biological-BMI-paper_misclassification_biological-BMI-BothSex_summary.csv'
bmiDF = pd.read_csv(fileDir+fileName, dtype={'public_client_id': str}).set_index('public_client_id')

#Clean to handle easier in this notebook
bmiDF.columns = bmiDF.columns.str.replace('Base', '')
bmiDF = bmiDF.drop(columns=['Race'])#Race has NaN in this cohort
print('NaN in DF:', bmiDF.isnull().to_numpy().sum(axis=None))
display(bmiDF)

In [ ]:
tempL = ['MetBMI', 'ProtBMI', 'ChemBMI', 'CombiBMI']
for bbmi in tempL:
    #Calculate difference rate
    bmiDF['diff_'+bbmi] = (bmiDF[bbmi] - bmiDF['BMI']) / bmiDF['BMI'] * 100
    print('Skewness of '+bbmi+' difference:', stats.skew(bmiDF['diff_'+bbmi]))
display(bmiDF)

In [ ]:
#Import the cleaned DFs of numeric features
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_data-cleaning_OLS-regression_numDF.tsv'
numDF = pd.read_csv(fileDir+fileName, sep='\t', dtype={'public_client_id': str}).set_index('public_client_id')

#Clean to handle easier in this notebook
numDF = numDF.rename(columns={'BaseAge':'Age'})
numDF = numDF.drop(columns=['Race'])#Race has NaN in this cohort
display(numDF)

> **–> In this version, eliminate 'activities.calories' here, which was to be eliminated in data cleaning.**  

In [ ]:
#Eliminate the exceptional feature
feature = 'activities.calories'
numDF = numDF.drop(columns=[feature])
display(numDF)

In [ ]:
#Define the list of covariates
covarL = ['Sex', 'Age', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5']

### 1-2. Split DF for each regression, and then drop NaN

> Sample size is different b/w numeric fetuares.  
> ***–> To maximize the sample size for each regression, dropping NaN is performed after splitting.***  

In [ ]:
featureL = numDF.drop(columns=covarL).columns.tolist()
tempDF = bmiDF.loc[:, ~bmiDF.columns.str.contains('_class')]

#Split DF for each regression and drop NaN
DF_splitL = []
for feature in featureL:
    tempDF1 = pd.merge(tempDF, numDF[feature], left_index=True, right_index=True, how='inner')
    DF_splitL.append(tempDF1.dropna())

print('The number of features:', len(DF_splitL))

In [ ]:
#Check examples for confirmation
print(featureL[0])
display(DF_splitL[0])
print(featureL[1])
display(DF_splitL[1])
print(featureL[-1])
display(DF_splitL[-1])

In [ ]:
#Check each sample size
tempL = []
for feature_i in range(len(featureL)):
    tempL.append(len(DF_splitL[feature_i]))
tempS = pd.Series(tempL, index=featureL)
tempS = tempS.sort_values(ascending=False)

display(tempS)
display(tempS.describe())

#Distribution
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
sns.distplot(tempS)
sns.despine()
plt.xlabel('Sample size')
plt.ylabel('Density')
plt.show()

### 1-3. Eliminate features with small sample size

> No need for further elimination in this study cohort!  

### 1-4. Standardization of continuous numeric features and covariates

> Because all features are continuous numeric features this time, age is also standardized.

In [ ]:
tempL = []
for feature_i in range(len(featureL)):
    tempDF = DF_splitL[feature_i].drop(columns=['Sex'])
    #Z-score transformation
    scaler = StandardScaler(copy=True, with_mean=True, with_std=True)
    tempA = scaler.fit_transform(tempDF)#Column direction
    tempDF = pd.DataFrame(data=tempA, index=tempDF.index, columns=tempDF.columns)
    #Add categorical covariates
    tempDF = pd.merge(tempDF, DF_splitL[feature_i]['Sex'], left_index=True, right_index=True)
    tempL.append(tempDF)
DF_splitL = tempL#Update/overwrite

In [ ]:
#Confirmation
tempL = []
for feature_i in range(len(featureL)):
    tempL.append(DF_splitL[feature_i].loc[:, featureL[feature_i]])
display(pd.concat(tempL, axis=1).describe())#Length is different but enforce merging to see the summary

#Check distribution of some example numeric features
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for feature_i in range(0, len(tempL), round(len(tempL)/4)):
    sns.distplot(tempL[feature_i], label=tempL[feature_i].name)
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

#Check distribution of BMI/bBMI and age in a example
tempDF = DF_splitL[0]
tempDF = tempDF.loc[:, (tempDF.columns.str.contains('log_')|tempDF.columns.str.contains('Age'))]
sns.set(style='ticks', font='Arial', context='notebook')
plt.figure(figsize=(4, 3))
for var in tempDF.columns.tolist():
    sns.distplot(tempDF[var], label=var)
sns.despine()
plt.xlabel('Z-score')
plt.ylabel('Density')
plt.legend(bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
plt.show()

### 1-5. One-hot encoding for categorical covariates

> category_encoders is more useful way than using sklearn.preprocessing or pandas.get_dummies.  
> ***–> In statsmodels, categorical variables are automatically recognized!***  
> –> Hence, this step is not needed anymore.

In [ ]:
#Check final DFs of examples for confirmation
print(featureL[0])
display(DF_splitL[0])
print(featureL[1])
display(DF_splitL[1])
print(featureL[-1])
display(DF_splitL[-1])

## 2. Version 2: (b)BMI ~ b0 + b1\*Feature + b2\*Sex + b3\*Age + b4\*AncestryPCs

> ***Main aim: Independently find associated features with BMI or biological BMI while adjusting sex, age and ancestry PCs as covariates***  

In [ ]:
tempL = ['BMI', 'MetBMI', 'ProtBMI', 'ChemBMI', 'CombiBMI']
olsDF = pd.DataFrame(index=featureL)
for bmi in tempL:
    #Perform OLS regressions
    tempL1 = []#For R2
    tempL2 = []#For beta-coefficient
    tempL3 = []#For SE of beta-coefficient
    tempL4 = []#For p-value
    tempL5 = []#For sample size
    tempL7 = []#For lower range of 95% CI for beta-coefficient
    tempL8 = []#For higher range of 95% CI for beta-coefficient
    t_start = time.time()
    for feature_i in range(len(featureL)):
        feature = featureL[feature_i]
        tempDF = DF_splitL[feature_i]
        tempL5.append(len(tempDF))#Same b/w bBMI in this cohort
        #Select dependent/independent variables
        tempL6 = [item for sublist in [['log_'+bmi, feature], covarL] for item in sublist]
        tempDF = tempDF[tempL6]
        tempDF = tempDF.rename(columns={'log_'+bmi:'log_BMI', feature:'Feature'})
        ##Add a constant for the intercept -> Similar to R, smf automatically add a constant
        #Fit model
        formula = 'log_BMI ~ Feature + C(Sex) + Age + PC1 + PC2 + PC3 + PC4 + PC5'
        resOLS = smf.ols(formula, data=tempDF).fit()
        #Save R2 [%]
        tempL1.append(resOLS.rsquared*100)
        #Save beta-coefficient of the variable
        tempL2.append(resOLS.params['Feature'])
        tempL3.append(resOLS.bse['Feature'])
        tempL7.append(resOLS.conf_int(alpha=0.05).loc['Feature', 0])
        tempL8.append(resOLS.conf_int(alpha=0.05).loc['Feature', 1])
        #Save p-value of the variable
        tempL4.append(resOLS.pvalues['Feature'])
    t_elapsed = time.time() - t_start
    print('Elapsed time for', len(featureL), 'OLS regressions of log_'+bmi+':',
          round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
    
    #Clean the results
    tempDF = pd.DataFrame({'R2_'+bmi:tempL1, 'Bcoef_'+bmi:tempL2, 'BcoefSE_'+bmi:tempL3,
                           'BcoefCIlow_'+bmi:tempL7, 'BcoefCIhigh_'+bmi:tempL8, 'Pval_'+bmi:tempL4},
                          index=featureL)
    ##P-value adjustment by using Benjamini–Hochberg method
    tempDF['BHPval_'+bmi] = multi.multipletests(tempDF['Pval_'+bmi], alpha=0.05, method='fdr_bh',
                                                is_sorted=False, returnsorted=False)[1]
    ##P-value adjustment by using Holm–Bonferroni method
    tempDF['HolmPval_'+bmi] = multi.multipletests(tempDF['Pval_'+bmi], alpha=0.05, method='holm',
                                                  is_sorted=False, returnsorted=False)[1]
    
    #Merge the cleaned results
    olsDF = pd.merge(olsDF, tempDF, left_index=True, right_index=True)

olsDF['SampleSize'] = tempL5
olsDF = olsDF.sort_values(by=['BHPval_BMI'], ascending=True)
olsDF.index.set_names('Feature', inplace=True)
display(olsDF)
#Save
fileDir = './ExportData/'
fileName = '220117_Biological-BMI-paper_OLS-regression_numeric-features-ver3_OLS2.tsv'
olsDF.to_csv(fileDir+fileName, sep='\t', index=True)

In [ ]:
#Significantly associated features
tempD = {'BMI':'0.3', 'MetBMI':'b', 'ProtBMI':'r', 'ChemBMI':'g', 'CombiBMI':'m'}
tempL = []
for bmi in list(tempD.keys()):
    #Extact significantly associated features
    tempDF = olsDF.loc[olsDF['BHPval_'+bmi]<0.05]
    print('The number of features significantly associated with '+bmi+' (FDR < 0.05):', len(tempDF))
    tempL.append(tempDF.index.tolist())
#Flatten and drop multiplicates
tempL = list(set(item for sublist in tempL for item in sublist))
tempDF = olsDF.loc[olsDF.index.isin(tempL)]
display(tempDF)

#Add sample size to feature label
tempDF = tempDF.reset_index()
tempDF['SampleSize'] = [f'{item:,}' for item in tempDF['SampleSize'].tolist()]
tempDF['Feature'] = tempDF['Feature'].str.cat(tempDF['SampleSize'], sep=' ('+r'$n$'+' = ')
tempDF['Feature'] = tempDF['Feature'].str.cat(np.repeat(')', len(tempDF)), sep='')
tempDF = tempDF.set_index('Feature')

#Visualize R2
tempDF1 = tempDF[['R2_'+item for item in list(tempD.keys())]].sort_values(by=['R2_BMI'], ascending=False)
tempL = tempDF1.index.tolist()#Save order
tempDF1 = tempDF1.reset_index().melt(var_name='BMItype', value_name='R2', id_vars=['Feature'])
tempDF1['BMItype'] = tempDF1['BMItype'].str.replace('R2_', '')
##Style and annotation info
tempL1 = []
tempL2 = []
for row_i in range(len(tempDF1)):
    bmi = tempDF1.iloc[row_i]['BMItype']
    feature = tempDF1.iloc[row_i]['Feature']
    #Association
    if tempDF.loc[feature, 'Bcoef_'+bmi]>0:
        tempL1.append('Positive association')
    elif tempDF.loc[feature, 'Bcoef_'+bmi]<0:
        tempL1.append('Negative association')
    else:#just in case
        tempL1.append('No association')
    #Significance
    if tempDF.loc[feature, 'BHPval_'+bmi]<0.05:
        tempL2.append('FDR<0.05')
    else:
        tempL2.append('n.s.')
tempDF1['Association'] = tempL1
tempDF1['Signif'] = tempL2
##Plot
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(7.5, 18))
p = sns.barplot(data=tempDF1, x='R2', y='Feature', order=tempL, hue='BMItype', hue_order=tempD.keys(),
                palette=tempD, dodge=True, edgecolor='black', linewidth=1)
p.grid(axis='x', linestyle='--', color='black')
for rect, row_i in zip(p.patches, range(len(tempDF1))):
    xcoord = rect.get_width()
    ycoord = rect.get_y() + rect.get_height()/2
    #Add annotation
    label = tempDF1.iloc[row_i]['Signif']
    bmi = tempDF1.iloc[row_i]['BMItype']
    if label=='n.s.':
        text_offset = 50
        hue_order = list(tempD.keys()).index(bmi)
        if xcoord > 0:
            if hue_order%2 == 0:
                offset = +text_offset/2
            else:
                offset = +text_offset
            halign = 'left'
        else:
            if hue_order%2 == 0:
                offset = -text_offset/2
            else:
                offset = -text_offset
            halign = 'right'
        p.annotate(label, xy=(xcoord, ycoord), xytext=(offset, 0), textcoords='offset points',
                   horizontalalignment=halign, verticalalignment='center',
                   fontsize='x-small', color='black',
                   arrowprops={'arrowstyle':'-', 'color':tempD[bmi], 'linewidth':1,
                               'shrinkA':offset/25, 'shrinkB':offset/5})
sns.despine()
plt.xlabel('Ratio of explained variance [%]')
plt.ylabel('')
for ycoord in range(len(tempDF)):
    if ycoord%2 == 0:
        plt.axhspan(ymin=ycoord-0.5, ymax=ycoord+0.5, facecolor='k', alpha=0.2)
plt.margins(0.02, 0.005, tight=True)
plt.legend(loc='lower right')
plt.show()

#Visualize beta-coefficient
tempDF1 = tempDF[['Bcoef_'+item for item in list(tempD.keys())]].sort_values(by=['Bcoef_BMI'], ascending=False)
tempL = tempDF1.index.tolist()#Save order
tempDF1 = tempDF1.reset_index().melt(var_name='BMItype', value_name='Bcoef', id_vars=['Feature'])
tempDF1['BMItype'] = tempDF1['BMItype'].str.replace('Bcoef_', '')
##Annotation info
tempL1 = []
tempL2 = []
for row_i in range(len(tempDF1)):
    bmi = tempDF1.iloc[row_i]['BMItype']
    feature = tempDF1.iloc[row_i]['Feature']
    #SE
    tempL1.append(tempDF.loc[feature, 'BcoefSE_'+bmi])
    #Significance
    if tempDF.loc[feature, 'BHPval_'+bmi]<0.05:
        tempL2.append('FDR<0.05')
    else:
        tempL2.append('n.s.')
tempDF1['BcoefSE'] = tempL1
tempDF1['Signif'] = tempL2
##Plot
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(7.5, 18))
p = sns.barplot(data=tempDF1, x='Bcoef', y='Feature', order=tempL, hue='BMItype', hue_order=tempD.keys(),
                palette=tempD, dodge=True, edgecolor='white', linewidth=0.5, saturation=1)
p.grid(axis='x', linestyle='--', color='black')
for rect, row_i in zip(p.patches, range(len(tempDF1))):
    xcoord = rect.get_width()
    ycoord = rect.get_y() + rect.get_height()/2
    #Add SEM
    sem = tempDF1.iloc[row_i]['BcoefSE']
    plt.errorbar(x=xcoord, y=ycoord, xerr=sem, fmt='none', ecolor='k', elinewidth=1, capsize=2)
    #Add annotation
    label = tempDF1.iloc[row_i]['Signif']
    bmi = tempDF1.iloc[row_i]['BMItype']
    if label=='n.s.':
        text_offset = 50
        hue_order = list(tempD.keys()).index(bmi)
        if xcoord > 0:
            if hue_order%2 == 0:
                offset = +text_offset/2
            else:
                offset = +text_offset
            halign = 'left'
            xcoord = xcoord + sem
        else:
            if hue_order%2 == 0:
                offset = -text_offset/2
            else:
                offset = -text_offset
            halign = 'right'
            xcoord = xcoord - sem
        p.annotate(label, xy=(xcoord, ycoord), xytext=(offset, 0), textcoords='offset points',
                   horizontalalignment=halign, verticalalignment='center',
                   fontsize='x-small', color='black',
                   arrowprops={'arrowstyle':'-', 'color':tempD[bmi], 'linewidth':1,
                               'shrinkA':offset/25, 'shrinkB':offset/5})
sns.despine()
plt.xlabel(r'$\beta$'+'-coefficient')
plt.ylabel('')
for ycoord in range(len(tempDF)):
    if ycoord%2 == 0:
        plt.axhspan(ymin=ycoord-0.5, ymax=ycoord+0.5, facecolor='k', alpha=0.2)
plt.margins(0.02, 0.005, tight=True)
plt.legend(loc='lower right')
plt.show()

In [ ]:
#Significantly associated features
tempD = {'BMI':'0.3', 'MetBMI':'b', 'ProtBMI':'r', 'ChemBMI':'g', 'CombiBMI':'m'}
tempL = []
for bmi in list(tempD.keys()):
    #Extact significantly associated features
    tempDF = olsDF.loc[olsDF['BHPval_'+bmi]<0.05]
    print('The number of features significantly associated with '+bmi+' (FDR < 0.05):', len(tempDF))
    tempL.append(tempDF.index.tolist())
#Flatten and drop multiplicates
tempL = list(set(item for sublist in tempL for item in sublist))
tempDF = olsDF.loc[olsDF.index.isin(tempL)]

#Shorten feature label
fileDir = './ImportData/'
fileName = '210727_feature-label-correspondence.csv'
tempDF1 = pd.read_csv(fileDir+fileName)
tempDF1 = tempDF1.rename(columns={'OriginalName':'Feature', 'LabelForFigure':'FeatureLabel'})
tempDF1 = tempDF1.set_index('Feature')
tempDF = pd.merge(tempDF, tempDF1, left_index=True, right_index=True, how='left')
display(tempDF)

#Add sample size to feature label
tempDF = tempDF.reset_index()
tempDF['SampleSize'] = [f'{item:,}' for item in tempDF['SampleSize'].tolist()]
tempDF['FeatureLabel'] = tempDF['FeatureLabel'].str.cat(tempDF['SampleSize'], sep=' ('+r'$n$'+' = ')
tempDF['FeatureLabel'] = tempDF['FeatureLabel'].str.cat(np.repeat(')', len(tempDF)), sep='')
tempDF = tempDF.set_index('FeatureLabel')

#Visualize beta-coefficient with forest plot
##Sort feature
tempDF1 = tempDF[['Bcoef_'+item for item in tempD.keys()]].sort_values(by=['Bcoef_BMI'], ascending=False)
tempL = tempDF1.index.tolist()#Save order
##Clean DF for plot
tempDF1 = tempDF.reset_index().melt(var_name='BMItype', value_name='Bcoef', id_vars=['FeatureLabel'],
                                    value_vars=['Bcoef_'+item for item in tempD.keys()])
tempDF1['BMItype'] = tempDF1['BMItype'].str.replace('Bcoef_', '')
tempDF2 = tempDF.reset_index().melt(var_name='BMItype', value_name='BcoefCIlow', id_vars=['FeatureLabel'],
                                    value_vars=['BcoefCIlow_'+item for item in tempD.keys()])
tempDF2['BMItype'] = tempDF2['BMItype'].str.replace('BcoefCIlow_', '')
tempDF1 = pd.merge(tempDF1, tempDF2, on=['FeatureLabel', 'BMItype'], how='left')
tempDF2 = tempDF.reset_index().melt(var_name='BMItype', value_name='BcoefCIhigh', id_vars=['FeatureLabel'],
                                    value_vars=['BcoefCIhigh_'+item for item in tempD.keys()])
tempDF2['BMItype'] = tempDF2['BMItype'].str.replace('BcoefCIhigh_', '')
tempDF1 = pd.merge(tempDF1, tempDF2, on=['FeatureLabel', 'BMItype'], how='left')
##Convert CI points to difference values
tempDF1['BcoefCIlow'] = tempDF1['Bcoef'] - tempDF1['BcoefCIlow']
tempDF1['BcoefCIhigh'] = tempDF1['BcoefCIhigh'] - tempDF1['Bcoef']
##Significance annotation
tempL1 = []
for row_i in range(len(tempDF1)):
    bmi = tempDF1['BMItype'].iloc[row_i]
    feature = tempDF1['FeatureLabel'].iloc[row_i]
    #Significance
    if tempDF.loc[feature, 'BHPval_'+bmi]<0.001:
        tempL1.append('***')
    elif tempDF.loc[feature, 'BHPval_'+bmi]<0.01:
        tempL1.append('**')
    elif tempDF.loc[feature, 'BHPval_'+bmi]<0.05:
        tempL1.append('*')
    else:
        tempL1.append('n.s.')
tempDF1['Signif'] = tempL1
##Color for edge
tempDF1['EdgeColor'] = tempDF1['BMItype'].map(tempD)
##Plot
ncols = 2
sns.set(style='ticks', font='Arial', context='talk')
fig, axes = plt.subplots(nrows=1, ncols=ncols, figsize=(16, 11), sharex=False, sharey=False)
for ax_i, ax in enumerate(axes.flat):
    half = int(len(tempL)/ncols)
    tempL1 = tempL[half*ax_i:half*(ax_i+1)]
    tempDF2 = tempDF1.loc[tempDF1['FeatureLabel'].isin(tempL1)].reset_index()
    #Point
    sns.pointplot(data=tempDF2, x='Bcoef', y='FeatureLabel', order=tempL1,
                  hue='BMItype', hue_order=tempD.keys(), palette=tempD,
                  dodge=0.7, join=False, ci=None, markers='o', scale=0.6, ax=ax)
    #Get coordinate of each point
    xcoordL = []
    ycoordL = []
    for coord in ax.collections:
        for x, y in coord.get_offsets():
            xcoordL.append(x)
            ycoordL.append(y)
    #Add errorbars and annotation
    for point_i in range(len(tempDF2)):
        #Be careful about order of ax.collections: tempL -> tempD.keys()
        label = tempL1[point_i%len(tempL1)]
        bmi = list(tempD.keys())[point_i//len(tempL1)]
        row_i = tempDF2.loc[(tempDF2['FeatureLabel']==label)&(tempDF2['BMItype']==bmi)].index.tolist()[0]
        #Add errorbars manually
        ax.errorbar(x=xcoordL[point_i], y=ycoordL[point_i],
                    xerr=[[tempDF2['BcoefCIlow'].iloc[row_i]], [tempDF2['BcoefCIhigh'].iloc[row_i]]],
                    fmt='', ecolor=tempDF2['EdgeColor'].iloc[row_i], elinewidth=4, capsize=0, capthick=4,
                    linestyle='', zorder=1)
        #Add significance
        signif = tempDF2['Signif'].iloc[row_i]
        if signif!='n.s.':
            text_offset = 5
            if xcoordL[point_i] > 0:
                xoffset = +text_offset
                halign = 'left'
                xcoord = xcoordL[point_i] + tempDF2['BcoefCIhigh'].iloc[row_i]
            else:
                xoffset = -text_offset
                halign = 'right'
                xcoord = xcoordL[point_i] - tempDF2['BcoefCIlow'].iloc[row_i]
            ycoord = ycoordL[point_i]
            yoffset = -4#Because asterisk looks upper shift
            ax.annotate(signif, xy=(xcoord, ycoord), xytext=(xoffset, yoffset), textcoords='offset points',
                        horizontalalignment=halign, verticalalignment='center',
                        fontsize='medium', color=tempD[bmi], zorder=2)
    #Add reference line
    #p.grid(axis='x', linestyle='--', color='k')
    ax.axvline(x=0, **{'linestyle':'--', 'color':'k', 'zorder':0})
    #Add shading
    for ycoord in range(len(tempL1)):
        if ycoord%2 == 0:
            ax.axhspan(ymin=ycoord-0.5, ymax=ycoord+0.5, facecolor='k', alpha=0.2, zorder=0)
    #Ax-dependent setting
    if ax_i == 0:
        #Range
        ax.set(xlim=(-0.15, 1.15), xticks=np.arange(0.0, 1.15, 0.5))
        #Add legend
        ax.get_legend().remove()
    elif ax_i ==1:
        #Range
        ax.set(xlim=(-0.65, 0.75), xticks=np.arange(-0.5, 0.75, 0.5))
        #Add legend
        ax.legend(loc='lower right')
sns.despine()
plt.setp(axes, ylim=(len(tempL1)-0.5, -0.5))#Otherwise, axes are extended
plt.setp(axes, xlabel=r'$\beta$'+'-coefficient', ylabel='')
fig.tight_layout()
##Save
fileDir = './ExportFigures/'
fileName = '220117_Biological-BMI-paper_OLS-regression_numeric-features_OLS2-bcoef.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

## 6. Categorical difference b/w bBMI and BMI based on categorical stratification

In [ ]:
#Misclassification
tempL = ['MetBMI', 'ProtBMI', 'ChemBMI', 'CombiBMI']
for bbmi in tempL:
    tempL1 = []
    for row_n in bmiDF.index.tolist():
        if bmiDF.loc[row_n, 'BMI_class'] == bmiDF.loc[row_n, bbmi+'_class']:
            tempL1.append('Matched')
        else:
            tempL1.append('Misclassified')
    bmiDF['vs_'+bbmi+'_class'] = tempL1
display(bmiDF)

### 6-1. Version 10-1 (for normal BMI class): Feature ~ b0 + b1\*Diff(bBMI – BMI) + b2\*Sex + b3\*Age + b4\*AncestryPCs

> ***Main aim: Find associated features with misclassification of BMI-based class while adjusting sex, age and ancestry PCs as covariates in the cohort of normal BMI class***  

> Features were already transformed to less skewed distribution; hence, able to use in linear regression. (cf.) This is same as t-test using GLM.  

In [ ]:
tempL = ['MetBMI', 'ProtBMI', 'ChemBMI', 'CombiBMI']
olsDF = pd.DataFrame(index=featureL)
for bmi in tempL:
    #Perform OLS regressions
    tempL1 = []#For sample size
    tempL2 = []#For residual degrees of freedom
    tempL3 = []#For t-static
    tempL4 = []#For p-value
    t_start = time.time()
    for feature_i in range(len(featureL)):
        feature = featureL[feature_i]
        tempDF = DF_splitL[feature_i]
        #Select the cohort
        tempDF = tempDF.loc[tempDF.index.isin(bmiDF.loc[bmiDF['BMI_class']=='Normal'].index)]
        tempL1.append(len(tempDF))#Same b/w bBMI in this cohort
        #Add misclassification label
        tempDF = pd.merge(tempDF, bmiDF['vs_'+bmi+'_class'], left_index=True, right_index=True)
        tempDF = tempDF.sort_values(by=['vs_'+bmi+'_class'], ascending=True)#To set 'Matched' as 0 in OLS
        #Select dependent/independent variables
        tempL5 = [item for sublist in [[feature, 'vs_'+bmi+'_class'], covarL] for item in sublist]
        tempDF = tempDF[tempL5]
        tempDF = tempDF.rename(columns={'vs_'+bmi+'_class':'Misclassification', feature:'Feature'})
        ##Add a constant for the intercept -> Similar to R, smf automatically add a constant
        #Fit model
        formula = 'Feature ~ C(Misclassification) + C(Sex) + Age + PC1 + PC2 + PC3 + PC4 + PC5'
        resOLS = smf.ols(formula, data=tempDF).fit()
        #Save residual df
        tempL2.append(resOLS.df_resid)
        #Save t-static of the variable
        tempL3.append(resOLS.tvalues['C(Misclassification)[T.Misclassified]'])
        #Save p-value of the variable
        tempL4.append(resOLS.pvalues['C(Misclassification)[T.Misclassified]'])
    t_elapsed = time.time() - t_start
    print('Elapsed time for', len(featureL), 'OLS regressions to misclassification with '+bmi+':',
          round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
    
    #Clean the results
    tempDF = pd.DataFrame({'df_'+bmi:tempL2, 't_'+bmi:tempL3, 'Pval_'+bmi:tempL4},
                          index=featureL)
    ##P-value adjustment by using Benjamini–Hochberg method
    tempDF['BHPval_'+bmi] = multi.multipletests(tempDF['Pval_'+bmi], alpha=0.05, method='fdr_bh',
                                                is_sorted=False, returnsorted=False)[1]
    ##P-value adjustment by using Holm–Bonferroni method
    tempDF['HolmPval_'+bmi] = multi.multipletests(tempDF['Pval_'+bmi], alpha=0.05, method='holm',
                                                  is_sorted=False, returnsorted=False)[1]
    
    #Merge the cleaned results
    olsDF = pd.merge(olsDF, tempDF, left_index=True, right_index=True)

olsDF['SampleSize'] = tempL1
olsDF = olsDF.sort_values(by=['BHPval_CombiBMI'], ascending=True)
olsDF.index.set_names('Feature', inplace=True)
display(olsDF)
#Save
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_OLS-regression_numeric-features_OLS10-normalBMI.tsv'
olsDF.to_csv(fileDir+fileName, sep='\t', index=True)

In [ ]:
#Significantly associated features
bmi_class='Normal'

#Prepare Z-score of each feature
tempDF = bmiDF[['BMI_class', 'vs_MetBMI_class', 'vs_ProtBMI_class', 'vs_ChemBMI_class', 'vs_CombiBMI_class']]
for feature_i in range(len(featureL)):
    feature = featureL[feature_i]
    tempS = DF_splitL[feature_i].loc[:, feature]
    tempDF = pd.merge(tempDF, tempS, left_index=True, right_index=True, how='outer')
tempDF = tempDF.loc[tempDF['BMI_class']==bmi_class]

tempD = {'MetBMI':'b', 'ProtBMI':'r', 'ChemBMI':'g', 'CombiBMI':'m'}
for bmi in list(tempD.keys()):
    print('vs. '+bmi)
    #Extact significantly associated features
    tempDF1 = olsDF.loc[olsDF['HolmPval_'+bmi]<0.05]
    print('The number of features significantly associated with misclassification with '+bmi+' (FWER < 0.05):',
          len(tempDF1))
    #Prepare feature label with sample size
    tempDF1['FeatureLabel'] = tempDF1.index.tolist()
    tempDF1['SampleSize'] = [f'{item:,}' for item in tempDF1['SampleSize'].tolist()]
    tempS = tempDF1['FeatureLabel'].str.cat(tempDF1['SampleSize'], sep=' ('+r'$n$'+' = ')
    tempS = tempS.str.cat(np.repeat(')', len(tempS)), sep='')
    #Prepare Z-score of significantly associated feature
    tempDF1 = tempDF.reset_index().melt(var_name='Feature', value_name='Zscore',
                                        value_vars=tempS.index.tolist(),
                                        id_vars=['public_client_id', 'vs_'+bmi+'_class'])
    tempDF1 = tempDF1.dropna()
    tempDF1 = pd.merge(tempDF1, tempS, left_on='Feature', right_index=True, how='left')
    #Plot
    tempD1 = {'Matched':'0.3', 'Misclassified':tempD[bmi]}
    tempD2 = {'MetBMI':'vs. Metabolomics', 'ProtBMI':'vs. Proteomics',
              'ChemBMI':'vs. Clinical labs', 'CombiBMI':'vs. Combined omics'}
    sns.set(style='ticks', font='Arial', context='talk')
    plt.figure(figsize=(5, 0.5*len(tempDF1['FeatureLabel'].unique())))
    p = sns.boxplot(data=tempDF1, y='FeatureLabel', x='Zscore', hue='vs_'+bmi+'_class',
                    hue_order=tempD1.keys(), dodge=True, palette=tempD1,
                    showfliers=True, flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4},
                    showcaps=True, notch=True)
    p.grid(axis='x', linestyle='--', color='black')
    sns.despine()
    plt.ylabel('')
    plt.xlabel(r'$Z$'+'-score')
    for row_i in range(len(tempS)):
        if row_i%2 == 0:
            plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor=tempD[bmi], alpha=0.2)
#    plt.margins(0.02, 0.005, tight=True)
    plt.legend(title=tempD2[bmi], bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
    plt.show()
    print('')

### 6-2. Version 10-2 (for obese BMI class): Feature ~ b0 + b1\*Diff(bBMI – BMI) + b2\*Sex + b3\*Age + b4\*AncestryPCs

> ***Main aim: Find associated features with misclassification of BMI-based class while adjusting sex, age and ancestry PCs as covariates in the cohort of obese BMI class***  

> Features were already transformed to less skewed distribution; hence, able to use in linear regression. (cf.) This is same as t-test using GLM.  

In [ ]:
tempL = ['MetBMI', 'ProtBMI', 'ChemBMI', 'CombiBMI']
olsDF = pd.DataFrame(index=featureL)
for bmi in tempL:
    #Perform OLS regressions
    tempL1 = []#For sample size
    tempL2 = []#For residual degrees of freedom
    tempL3 = []#For t-static
    tempL4 = []#For p-value
    t_start = time.time()
    for feature_i in range(len(featureL)):
        feature = featureL[feature_i]
        tempDF = DF_splitL[feature_i]
        #Select the cohort
        tempDF = tempDF.loc[tempDF.index.isin(bmiDF.loc[bmiDF['BMI_class']=='Obese'].index)]
        tempL1.append(len(tempDF))#Same b/w bBMI in this cohort
        #Add misclassification label
        tempDF = pd.merge(tempDF, bmiDF['vs_'+bmi+'_class'], left_index=True, right_index=True)
        tempDF = tempDF.sort_values(by=['vs_'+bmi+'_class'], ascending=True)#To set 'Matched' as 0 in OLS
        #Select dependent/independent variables
        tempL5 = [item for sublist in [[feature, 'vs_'+bmi+'_class'], covarL] for item in sublist]
        tempDF = tempDF[tempL5]
        tempDF = tempDF.rename(columns={'vs_'+bmi+'_class':'Misclassification', feature:'Feature'})
        ##Add a constant for the intercept -> Similar to R, smf automatically add a constant
        #Fit model
        formula = 'Feature ~ C(Misclassification) + C(Sex) + Age + PC1 + PC2 + PC3 + PC4 + PC5'
        resOLS = smf.ols(formula, data=tempDF).fit()
        #Save residual df
        tempL2.append(resOLS.df_resid)
        #Save t-static of the variable
        tempL3.append(resOLS.tvalues['C(Misclassification)[T.Misclassified]'])
        #Save p-value of the variable
        tempL4.append(resOLS.pvalues['C(Misclassification)[T.Misclassified]'])
    t_elapsed = time.time() - t_start
    print('Elapsed time for', len(featureL), 'OLS regressions to misclassification with '+bmi+':',
          round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')
    
    #Clean the results
    tempDF = pd.DataFrame({'df_'+bmi:tempL2, 't_'+bmi:tempL3, 'Pval_'+bmi:tempL4},
                          index=featureL)
    ##P-value adjustment by using Benjamini–Hochberg method
    tempDF['BHPval_'+bmi] = multi.multipletests(tempDF['Pval_'+bmi], alpha=0.05, method='fdr_bh',
                                                is_sorted=False, returnsorted=False)[1]
    ##P-value adjustment by using Holm–Bonferroni method
    tempDF['HolmPval_'+bmi] = multi.multipletests(tempDF['Pval_'+bmi], alpha=0.05, method='holm',
                                                  is_sorted=False, returnsorted=False)[1]
    
    #Merge the cleaned results
    olsDF = pd.merge(olsDF, tempDF, left_index=True, right_index=True)

olsDF['SampleSize'] = tempL1
olsDF = olsDF.sort_values(by=['BHPval_CombiBMI'], ascending=True)
olsDF.index.set_names('Feature', inplace=True)
display(olsDF)
#Save
fileDir = './ExportData/'
fileName = '210106_Biological-BMI-paper_OLS-regression_numeric-features_OLS10-obeseBMI.tsv'
olsDF.to_csv(fileDir+fileName, sep='\t', index=True)

In [ ]:
#Significantly associated features
bmi_class='Obese'

#Prepare Z-score of each feature
tempDF = bmiDF[['BMI_class', 'vs_MetBMI_class', 'vs_ProtBMI_class', 'vs_ChemBMI_class', 'vs_CombiBMI_class']]
for feature_i in range(len(featureL)):
    feature = featureL[feature_i]
    tempS = DF_splitL[feature_i].loc[:, feature]
    tempDF = pd.merge(tempDF, tempS, left_index=True, right_index=True, how='outer')
tempDF = tempDF.loc[tempDF['BMI_class']==bmi_class]

tempD = {'MetBMI':'b', 'ProtBMI':'r', 'ChemBMI':'g', 'CombiBMI':'m'}
for bmi in list(tempD.keys()):
    print('vs. '+bmi)
    #Extact significantly associated features
    tempDF1 = olsDF.loc[olsDF['HolmPval_'+bmi]<0.05]
    print('The number of features significantly associated with misclassification with '+bmi+' (FWER < 0.05):',
          len(tempDF1))
    #Prepare feature label with sample size
    tempDF1['FeatureLabel'] = tempDF1.index.tolist()
    tempDF1['SampleSize'] = [f'{item:,}' for item in tempDF1['SampleSize'].tolist()]
    tempS = tempDF1['FeatureLabel'].str.cat(tempDF1['SampleSize'], sep=' ('+r'$n$'+' = ')
    tempS = tempS.str.cat(np.repeat(')', len(tempS)), sep='')
    #Prepare Z-score of significantly associated feature
    tempDF1 = tempDF.reset_index().melt(var_name='Feature', value_name='Zscore',
                                        value_vars=tempS.index.tolist(),
                                        id_vars=['public_client_id', 'vs_'+bmi+'_class'])
    tempDF1 = tempDF1.dropna()
    tempDF1 = pd.merge(tempDF1, tempS, left_on='Feature', right_index=True, how='left')
    #Plot
    tempD1 = {'Matched':'0.3', 'Misclassified':tempD[bmi]}
    tempD2 = {'MetBMI':'vs. Metabolomics', 'ProtBMI':'vs. Proteomics',
              'ChemBMI':'vs. Clinical labs', 'CombiBMI':'vs. Combined omics'}
    sns.set(style='ticks', font='Arial', context='talk')
    plt.figure(figsize=(5, 0.5*len(tempDF1['FeatureLabel'].unique())))
    p = sns.boxplot(data=tempDF1, y='FeatureLabel', x='Zscore', hue='vs_'+bmi+'_class',
                    hue_order=tempD1.keys(), dodge=True, palette=tempD1,
                    showfliers=True, flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4},
                    showcaps=True, notch=True)
    p.grid(axis='x', linestyle='--', color='black')
    sns.despine()
    plt.ylabel('')
    plt.xlabel(r'$Z$'+'-score')
    for row_i in range(len(tempS)):
        if row_i%2 == 0:
            plt.axhspan(ymin=row_i-0.5, ymax=row_i+0.5, facecolor=tempD[bmi], alpha=0.2)
#    plt.margins(0.02, 0.005, tight=True)
    plt.legend(title=tempD2[bmi], bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
    plt.show()
    print('')

### 6-3. For presentation

> Because I couldn't generate “neat" Facet style in seaborn/matplotlib, each panel must be combined using image editor later.  

In [ ]:
tempL = ['waist-to-height_ratio', 'heartrate.resting',
         'MEAN_ARTERIAL_BLOOD_PRESSURE', 'activities.floors', 'polygenic_bmi_2018_11_raw']
margin = 0.5
for feature in tempL:
    #Prepare Z-score of each feature
    tempDF = bmiDF[['BMI_class', 'vs_MetBMI_class', 'vs_ProtBMI_class', 'vs_ChemBMI_class', 'vs_CombiBMI_class']]
    feature_i = featureL.index(feature)
    tempS = DF_splitL[feature_i].loc[:, feature]
    tempDF = pd.merge(tempDF, tempS, left_index=True, right_index=True, how='outer')
    tempDF = tempDF.dropna()
    print(feature, '(n = ', len(tempDF), ')')
    
    #Plot overall
    tempDF['xLabel'] = np.repeat('Overall', len(tempDF))
    tempD = {'Underweight':'blue', 'Normal':'green', 'Overweight':'orange', 'Obese':'red'}
    sns.set(style='ticks', font='Arial', context='talk')
    plt.figure(figsize=(1.5, 3))
    p = sns.boxplot(data=tempDF, y=feature, x='xLabel', hue='BMI_class',
                    hue_order=tempD.keys(), dodge=True, palette=tempD,
                    showfliers=False,#flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4},
                    showcaps=True, notch=True)
    p.set(xlim=(0-margin, 0+margin), ylim=(-3.2, 3.8), yticks=np.arange(-2, 2.1, 2))
    p.grid(axis='y', linestyle='--', color='black')
    lines = p.axes.get_lines()#Line2D: [[Q1, Q1-1.5IQR], [Q3, Q3+1.5IQR], [Q1, Q1], [Q3, Q3], [Med, Med], [flier]]
    for line in lines:
        line.set_color('k')
    for box in p.artists:
        box.set_edgecolor('k')
    sns.despine()
    plt.xlabel('')
    plt.ylabel(r'$Z$'+'-score')
    plt.legend(title='BMI class', bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
    ##Save
    fileDir = './ExportFigures/'
    fileName = '210106_Biological-BMI-paper_OLS-regression_numeric-features_('+feature+')-Overall.tif'
    plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                      pil_kwargs={'compression':'tiff_lzw'})
    plt.show()
    
    #Plot misclassification
    tempD = {'Normal':'green', 'Obese':'red'}
    tempDF = tempDF.loc[tempDF['BMI_class'].isin(list(tempD.keys()))]
    tempD1 = {'MetBMI':'b', 'ProtBMI':'r'}
    for bmi in list(tempD1.keys()):
        tempD2 = {'Matched':'0.8', 'Misclassified':tempD1[bmi]}
        tempD3 = {'MetBMI':'vs. Metabolomics', 'ProtBMI':'vs. Proteomics',
                  'ChemBMI':'vs. Clinical labs', 'CombiBMI':'vs. Combined omics'}
        #Retrieve statistical significance
        tempS = pd.Series(name='Pval')
        for bmi_class in list(tempD.keys()):
            fileDir = './ExportData/'
            fileName = '210106_Biological-BMI-paper_OLS-regression_numeric-features_OLS10-'+bmi_class.lower()+'BMI.tsv'
            olsDF = pd.read_csv(fileDir+fileName, sep='\t').set_index('Feature')
            pval = olsDF.loc[feature, 'Pval_'+bmi]
            if pval<0.001:
                tempS.loc[bmi_class] = '***'
            elif pval<0.01:
                tempS.loc[bmi_class] = '**'
            elif pval<0.05:
                tempS.loc[bmi_class] = '*'
            else:
                tempS.loc[bmi_class] = 'n.s.'
        #Plot
        sns.set(style='ticks', font='Arial', context='talk')
        plt.figure(figsize=(2, 3))
        p = sns.boxplot(data=tempDF, y=feature, x='BMI_class', order=tempD.keys(),
                        hue='vs_'+bmi+'_class', hue_order=tempD2.keys(), dodge=True, palette=tempD2,
                        showfliers=False,#flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4},
                        showcaps=True, notch=True)
        p.set(xlim=(0-margin, (len(tempD)-1)+margin), ylim=(-3.2, 3.8), yticks=np.arange(-2, 2.1, 2))
        p.grid(axis='y', linestyle='--', color='black')
        lines = p.axes.get_lines()#Line2D: [[Q1, Q1-1.5IQR], [Q3, Q3+1.5IQR], [Q1, Q1], [Q3, Q3], [Med, Med], [flier]]
        lines_unit = 5 + int(False)#showfliers=False
        for class_i in range(len(tempD)):
            #Matched
            whisker_0 = lines[class_i*lines_unit*len(tempD2) + lines_unit*0 + 1]
            xcoord_0 = whisker_0._x[1]#Q3+1.5IQR
            ycoord_0 = whisker_0._y[1]#Q3+1.5IQR
            #Misclassified
            whisker_1 = lines[class_i*lines_unit*len(tempD2) + lines_unit*1 + 1]
            xcoord_1 = whisker_1._x[1]#Q3+1.5IQR
            ycoord_1 = whisker_1._y[1]#Q3+1.5IQR
            #Standard point for annotation
            xcoord = (xcoord_0+xcoord_1)/2
            ycoord = max(ycoord_0, ycoord_1)
            label = tempS.iloc[class_i]
            if label!='n.s.':
                #Add annotation lines
                aline_offset = 0.2
                aline_length = 0.2 + aline_offset
                plt.plot([xcoord_0, xcoord_0, xcoord_1, xcoord_1],
                         [ycoord+aline_offset, ycoord+aline_length, ycoord+aline_length, ycoord+aline_offset],
                         lw=1.5, c='k')
                #Add annotation text
                text_offset = 0.2
                p.annotate(label, xy=(xcoord, ycoord), xytext=(0, text_offset), textcoords='offset points',
                           horizontalalignment='center', verticalalignment='bottom',
                           fontsize='medium', color='k')
        for line in lines:
            line.set_color('k')
        for box in p.artists:
            box.set_edgecolor('k')
        sns.despine()
        plt.xlabel('')
        plt.ylabel(r'$Z$'+'-score')
        plt.legend(title=tempD3[bmi], bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
        ##Save
        fileDir = './ExportFigures/'
        fileName = '210106_Biological-BMI-paper_OLS-regression_numeric-features_('+feature+')-'+bmi+'.tif'
        plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                          pil_kwargs={'compression':'tiff_lzw'})
        plt.show()
    
    print('')

In [ ]:
#As another negative control, age is prepared
margin = 0.5
#Prepare Z-score of age
tempL = ['BMI_class', 'vs_MetBMI_class', 'vs_ProtBMI_class', 'vs_ChemBMI_class', 'vs_CombiBMI_class']
tempL = [item for sublist in [tempL, covarL] for item in sublist]
tempDF = bmiDF[tempL]
tempS = tempDF['Age'].copy()
tempDF['Age'] = (tempS - tempS.mean())/tempS.std(ddof=0)#sklearn.preprocessing.StandardScaler uses ddof=0
tempDF = tempDF.dropna()
print('Age (n = ', len(tempDF), ')')

#Plot overall
tempDF['xLabel'] = np.repeat('Overall', len(tempDF))
tempD = {'Underweight':'blue', 'Normal':'green', 'Overweight':'orange', 'Obese':'red'}
sns.set(style='ticks', font='Arial', context='talk')
plt.figure(figsize=(1.5, 3))
p = sns.boxplot(data=tempDF, y='Age', x='xLabel', hue='BMI_class',
                hue_order=tempD.keys(), dodge=True, palette=tempD,
                showfliers=False,#flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4},
                showcaps=True, notch=True)
p.set(xlim=(0-margin, 0+margin), ylim=(-3.2, 3.8), yticks=np.arange(-2, 2.1, 2))
p.grid(axis='y', linestyle='--', color='black')
lines = p.axes.get_lines()#Line2D: [[Q1, Q1-1.5IQR], [Q3, Q3+1.5IQR], [Q1, Q1], [Q3, Q3], [Med, Med], [flier]]
for line in lines:
    line.set_color('k')
for box in p.artists:
    box.set_edgecolor('k')
sns.despine()
plt.xlabel('')
plt.ylabel(r'$Z$'+'-score')
plt.legend(title='BMI class', bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
##Save
fileDir = './ExportFigures/'
fileName = '210106_Biological-BMI-paper_OLS-regression_numeric-features_('+'Age'+')-Overall.tif'
plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                  pil_kwargs={'compression':'tiff_lzw'})
plt.show()

#Plot misclassification
tempD = {'Normal':'green', 'Obese':'red'}
tempDF = tempDF.loc[tempDF['BMI_class'].isin(list(tempD.keys()))]
tempD1 = {'MetBMI':'b', 'ProtBMI':'r'}
for bmi in list(tempD1.keys()):
    tempD2 = {'Matched':'0.8', 'Misclassified':tempD1[bmi]}
    tempD3 = {'MetBMI':'vs. Metabolomics', 'ProtBMI':'vs. Proteomics',
              'ChemBMI':'vs. Clinical labs', 'CombiBMI':'vs. Combined omics'}
    #Calculate statistical significance for age with version 10 model (see 11-3 or 11-4)
    tempS = pd.Series(name='Pval')
    for bmi_class in list(tempD.keys()):
        #Select the cohort
        tempDF1 = tempDF.loc[tempDF['BMI_class']==bmi_class]
        tempDF1 = tempDF1.sort_values(by=['vs_'+bmi+'_class'], ascending=True)#To set 'Matched' as 0 in OLS
        #Select dependent/independent variables
        tempL = [item for sublist in [['vs_'+bmi+'_class'], covarL] for item in sublist]
        tempDF1 = tempDF1[tempL]
        tempDF1 = tempDF1.rename(columns={'vs_'+bmi+'_class':'Misclassification'})
        ##Add a constant for the intercept -> Similar to R, smf automatically add a constant
        #Fit model
        formula = 'Age ~ C(Misclassification) + C(Sex) + PC1 + PC2 + PC3 + PC4 + PC5'
        resOLS = smf.ols(formula, data=tempDF1).fit()
        #Save p-value of the variable
        pval = resOLS.pvalues['C(Misclassification)[T.Misclassified]']
        #Convet to label
        if pval<0.001:
            tempS.loc[bmi_class] = '***'
        elif pval<0.01:
            tempS.loc[bmi_class] = '**'
        elif pval<0.05:
            tempS.loc[bmi_class] = '*'
        else:
            tempS.loc[bmi_class] = 'n.s.'
    #Plot
    sns.set(style='ticks', font='Arial', context='talk')
    plt.figure(figsize=(2, 3))
    p = sns.boxplot(data=tempDF, y='Age', x='BMI_class', order=tempD.keys(),
                    hue='vs_'+bmi+'_class', hue_order=tempD2.keys(), dodge=True, palette=tempD2,
                    showfliers=False,#flierprops={'marker':'o', 'markerfacecolor':'gray', 'alpha':0.4},
                    showcaps=True, notch=True)
    p.set(xlim=(0-margin, (len(tempD)-1)+margin), ylim=(-3.2, 3.8), yticks=np.arange(-2, 2.1, 2))
    p.grid(axis='y', linestyle='--', color='black')
    lines = p.axes.get_lines()#Line2D: [[Q1, Q1-1.5IQR], [Q3, Q3+1.5IQR], [Q1, Q1], [Q3, Q3], [Med, Med], [flier]]
    lines_unit = 5 + int(False)#showfliers=False
    for class_i in range(len(tempD)):
        #Matched
        whisker_0 = lines[class_i*lines_unit*len(tempD2) + lines_unit*0 + 1]
        xcoord_0 = whisker_0._x[1]#Q3+1.5IQR
        ycoord_0 = whisker_0._y[1]#Q3+1.5IQR
        #Misclassified
        whisker_1 = lines[class_i*lines_unit*len(tempD2) + lines_unit*1 + 1]
        xcoord_1 = whisker_1._x[1]#Q3+1.5IQR
        ycoord_1 = whisker_1._y[1]#Q3+1.5IQR
        #Standard point for annotation
        xcoord = (xcoord_0+xcoord_1)/2
        ycoord = max(ycoord_0, ycoord_1)
        label = tempS.iloc[class_i]
        if label!='n.s.':
            #Add annotation lines
            aline_offset = 0.2
            aline_length = 0.2 + aline_offset
            plt.plot([xcoord_0, xcoord_0, xcoord_1, xcoord_1],
                     [ycoord+aline_offset, ycoord+aline_length, ycoord+aline_length, ycoord+aline_offset],
                     lw=1.5, c='k')
            #Add annotation text
            text_offset = 0.2
            p.annotate(label, xy=(xcoord, ycoord), xytext=(0, text_offset), textcoords='offset points',
                       horizontalalignment='center', verticalalignment='bottom',
                       fontsize='medium', color='k')
    for line in lines:
        line.set_color('k')
    for box in p.artists:
        box.set_edgecolor('k')
    sns.despine()
    plt.xlabel('')
    plt.ylabel(r'$Z$'+'-score')
    plt.legend(title=tempD3[bmi], bbox_to_anchor=(1, 0.5), loc='center left', borderaxespad=1)
    ##Save
    fileDir = './ExportFigures/'
    fileName = '210106_Biological-BMI-paper_OLS-regression_numeric-features_('+'Age'+')-'+bmi+'.tif'
    plt.gcf().savefig(fileDir+fileName, dpi=300, bbox_inches='tight', pad_inches=0,
                      pil_kwargs={'compression':'tiff_lzw'})
    plt.show()